# Phase 2 — Input Noise Sweep

**EECS 6699 Final Project · Day 1**

Train deep and shallow models on **clean** data, then evaluate them on inputs perturbed by
$\tilde{x} = \mathrm{clip}_{[0,1]}(x + \varepsilon),\ \varepsilon \sim \mathcal{N}(0, \sigma^2)$.

**Hypothesis H2.** For $\sigma \ll 2^{-k}$ the deep network retains its advantage; for
$\sigma \gtrsim 2^{-k}$ the gap closes and may invert. We expect a crossover $\sigma_x^\star \approx 2^{-k}$.
With $k=4$ that puts $\sigma_x^\star \approx 0.0625$.


## 0. Install dependencies

Run once per kernel. `%pip` installs into the *active* Jupyter kernel (safer than `!pip`).
Skip this cell if your environment already has these packages.


In [1]:
%pip install --quiet torch numpy matplotlib scipy



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.targets import sawtooth_target, make_dataset
from src.models  import make_matched_pair
from src.train   import TrainConfig, multi_seed_run
from src.noise   import evaluate_with_input_noise

FIG_DIR = ROOT / 'results' / 'figures'
TAB_DIR = ROOT / 'results' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
K = 4
SEEDS = [0, 1, 2, 3, 4]
SIGMAS = [0.0, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 0.0625, 1e-1, 2e-1]   # spans below and above 2^{-k}
N_REPEATS = 30
print('device:', device, '  expected sigma* =', 2.0 ** (-K))


device: cpu   expected sigma* = 0.0625


## 1. Train both models on clean data (5 seeds)


In [3]:
cfg = TrainConfig(
    epochs=30_000, lr=5e-3, n_train=1200, k=K, log_every=3_000, device=device,
    curriculum_ks=[1, 2, 3, 4],
    curriculum_weights=[1, 1, 2, 4],
    reset_lr_per_stage=True,
)
results = multi_seed_run(SEEDS, cfg, deep_depth=9, deep_width=8)

[seed 0] deep params = 529, shallow params = 529 (width = 176)
[seed 0] training deep ...
  ── stage 1/4: k=1, epochs=3750 ──
  step      0 k=1   loss = 3.360535e-01  lr = 5.00e-03
  step   3000 k=1   loss = 6.647207e-08  lr = 4.85e-04
  ── stage 2/4: k=2, epochs=3750 ──
  step   6000 k=2   loss = 5.926057e-08  lr = 1.73e-03
  ── stage 3/4: k=3, epochs=7500 ──
  step   9000 k=3   loss = 4.272682e-04  lr = 4.52e-03
  step  12000 k=3   loss = 6.025028e-07  lr = 1.73e-03
  ── stage 4/4: k=4, epochs=15000 ──
  step  15000 k=4   loss = 1.664420e-01  lr = 5.00e-03
  step  18000 k=4   loss = 3.221719e-02  lr = 4.52e-03
  step  21000 k=4   loss = 2.982992e-02  lr = 3.28e-03
  step  24000 k=4   loss = 2.861558e-02  lr = 1.73e-03
  step  27000 k=4   loss = 2.857817e-02  lr = 4.86e-04
[seed 0] training shallow ...
  ── stage 1/4: k=1, epochs=3750 ──
  step      0 k=1   loss = 1.504602e-01  lr = 5.00e-03
  step   3000 k=1   loss = 3.324450e-08  lr = 4.85e-04
  ── stage 2/4: k=2, epochs=3750 ──
  s

## 2. Sweep input-noise levels per model and seed

We re-use the clean targets `y_clean`; only `x` is perturbed at evaluation.


In [4]:
x_eval = torch.linspace(0, 1, 4000).view(-1, 1).to(device)
y_eval = sawtooth_target(x_eval, k=K)

sweep = {'deep': {s: [] for s in SIGMAS},
         'shallow': {s: [] for s in SIGMAS}}

for tag in ('deep', 'shallow'):
    for seed_idx, seed in enumerate(SEEDS):
        model = results[tag]['models'][seed_idx]
        per_sigma = evaluate_with_input_noise(
            model, x_eval, y_eval, sigmas=SIGMAS,
            n_repeats=N_REPEATS, seed=1000 + seed,
        )
        for s in SIGMAS:
            sweep[tag][s].append(per_sigma[s]['mean'])

sweep_arr = {tag: np.array([sweep[tag][s] for s in SIGMAS]) for tag in ('deep','shallow')}  # shape (n_sigma, n_seeds)
{tag: arr.shape for tag, arr in sweep_arr.items()}


{'deep': (10, 5), 'shallow': (10, 5)}

## 3. Crossover analysis


In [5]:
def mean_std(a):
    return a.mean(axis=1), a.std(axis=1, ddof=0)

deep_mu, deep_sd = mean_std(sweep_arr['deep'])
shal_mu, shal_sd = mean_std(sweep_arr['shallow'])

# Find the smallest sigma at which deep_mu >= shallow_mu (advantage flips).
flip_idx = next((i for i, (d, s) in enumerate(zip(deep_mu, shal_mu)) if d >= s), None)
sigma_star = SIGMAS[flip_idx] if flip_idx is not None else None
print('crossover sigma* =', sigma_star, '   theoretical 2^{-k} =', 2.0 ** (-K))


crossover sigma* = 0.03    theoretical 2^{-k} = 0.0625


In [6]:
import csv
with open(TAB_DIR / 'phase2_input_noise.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['sigma','deep_mse_mean','deep_mse_std','shallow_mse_mean','shallow_mse_std'])
    for i, s in enumerate(SIGMAS):
        w.writerow([s, deep_mu[i], deep_sd[i], shal_mu[i], shal_sd[i]])
print('wrote', TAB_DIR / 'phase2_input_noise.csv')


wrote results/tables/phase2_input_noise.csv


## 4. Plot: MSE vs $\sigma$ (log-log) with seed-bands


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
xs = np.array(SIGMAS, dtype=float)
# Replace sigma=0 with a small offset for log axis; mark separately.
xs_plot = np.where(xs == 0, 1e-5, xs)

for mu, sd, color, label in [
    (deep_mu,  deep_sd,  'tab:green', 'Deep (L=9, W=8)'),
    (shal_mu,  shal_sd,  'tab:red',   'Shallow (L=2, matched)'),
]:
    ax.plot(xs_plot, mu, color=color, marker='o', label=label)
    ax.fill_between(xs_plot, np.maximum(mu - sd, 1e-12), mu + sd, color=color, alpha=0.18)

ax.set_xscale('log'); ax.set_yscale('log')
ax.axvline(2.0 ** (-K), color='black', linestyle=':', alpha=0.6, label=r'$2^{-k}$ (theory)')
if sigma_star is not None and sigma_star > 0:
    ax.axvline(sigma_star, color='gray', linestyle='--', alpha=0.6, label=fr'empirical $\sigma^\star$={sigma_star:g}')
ax.set_xlabel(r'input noise $\sigma$  (log scale; $\sigma=0$ shifted to $10^{-5}$ for display)')
ax.set_ylabel('test MSE on clean targets')
ax.set_title('Phase 2 — depth advantage erodes as input noise grows')
ax.grid(which='both', alpha=0.25); ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'phase2_input_noise.png', dpi=150)
plt.show()

## 5. Verdict on H2

H2 predicts that the deep advantage is preserved for $\sigma \ll 2^{-k}$ and erodes / inverts for
$\sigma \gtrsim 2^{-k}$. Compare the empirical $\sigma^\star$ printed above to the dashed vertical line at $2^{-k}$.


In [8]:
h2_supported = (sigma_star is not None) and (sigma_star <= 2 * 2.0 ** (-K))
print('H2 supported (sigma* within 2x of theory):', h2_supported)
summary = {
    'k': K,
    'theoretical_sigma_star': 2.0 ** (-K),
    'empirical_sigma_star':   sigma_star,
    'H2_supported':           bool(h2_supported),
    'sigmas': SIGMAS,
    'deep_mse_mean':    deep_mu.tolist(),
    'shallow_mse_mean': shal_mu.tolist(),
}
(TAB_DIR / 'phase2_summary.json').write_text(__import__('json').dumps(summary, indent=2))
summary


H2 supported (sigma* within 2x of theory): True


{'k': 4,
 'theoretical_sigma_star': 0.0625,
 'empirical_sigma_star': 0.03,
 'H2_supported': True,
 'sigmas': [0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.0625, 0.1, 0.2],
 'deep_mse_mean': [0.032569516173381885,
  0.03257100173291292,
  0.032583239927407706,
  0.03272032343666069,
  0.03389307097531855,
  0.04542374312877655,
  0.10230898559093475,
  0.13795050084590912,
  0.14355524331331254,
  0.15551456362009047],
 'shallow_mse_mean': [0.042107169330120084,
  0.04210767447948456,
  0.04211530387401581,
  0.04219742938876152,
  0.04287799075245857,
  0.04979953542351723,
  0.08445836752653121,
  0.10728362798690796,
  0.11205340176820755,
  0.12259965240955353]}